# Behavioral, Social Communication, and QEEG Measures from Children with Autism Spectrum Disorder Undergoing Low intensity rTMS Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is provided via a Croissant schema URL and adheres to FAIR standards.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL for FAIR^2
url = 'https://sen.science/doi/10.71728/senscience.4w1g-g9sj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")
print(f"Published: {metadata['datePublished']}")
print(f"License: {metadata['license']}")
print(f"Source identifier: {metadata['identifier']}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id` values (identifiers).

In [ ]:
# List all RecordSets with their @id
record_sets = dataset.metadata.record_set

if not record_sets:
    print("No record sets found in metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}, name: {rs.get('name', 'No name')}\nFields:")
        fields = rs.get('field', [])
        for field in fields:
            print(f"  Field @id: {field['@id']}, name: {field.get('name', 'No name')}, dataType: {field.get('dataType', 'Unknown')}")
        print("")
    print(f"Total record sets found: {len(record_sets)}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

If your dataset contains multiple record sets, we will loop through each and load into pandas DataFrames, referenced by `@id`.

In [ ]:
# Prepare to extract data from all record sets
dataframes = {}

# Collect record set @ids
record_set_ids = []
for rs in dataset.metadata.record_set:
    record_set_ids.append(rs['@id'])

# Main extraction loop
for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet @id: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for {record_set_id}: {df.columns.tolist()}")
        print(df.head(2))
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")
        dataframes[record_set_id] = None

# For demonstration, select the first record set with data
target_record_set_id = None
for rsid in record_set_ids:
    if isinstance(dataframes.get(rsid), pd.DataFrame) and not dataframes[rsid].empty:
        target_record_set_id = rsid
        break

if target_record_set_id:
    print(f"Selected record set for further analysis: {target_record_set_id}")
    print(f"Columns: {dataframes[target_record_set_id].columns.tolist()}")
    display(dataframes[target_record_set_id].head())
else:
    print("No valid record set found for extraction.")

## 4. Exploratory Data Analysis (EDA)
We'll apply data filtering, normalization, and grouping on the extracted data. All references use the entity `@id` identifiers for fields.

If numeric fields (columns) exist, we'll process them. Otherwise, we'll demonstrate with available columns.

In [ ]:
# EDA Step: Identify numeric field(s) by their @id
df = dataframes.get(target_record_set_id)
if df is not None and not df.empty:
    # Try to identify numeric columns by dtype
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        print(f"Using numeric field: {numeric_field_id}")

        # Filter for values above threshold
        threshold = df[numeric_field_id].mean()  # simple threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalization (z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized values for {numeric_field_id} (first few rows):")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by another field if present
        # Find first non-numeric column for grouping
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and (pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_object_dtype(df[col])):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(f"Grouped mean of {numeric_field_id} by {group_field} (first few groups):")
            print(grouped_df.head())
        else:
            print("No suitable string/object field found for grouping.")
    else:
        print("No numeric columns found in DataFrame.")
else:
    print("No DataFrame loaded for EDA.")

## 5. Visualization
Visualize the numeric field distribution and grouped statistics using plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization based on EDA results
if df is not None and not df.empty:
    if 'numeric_field_id' in locals():
        plt.figure(figsize=(7, 4))
        sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
        plt.title(f"Distribution of '{numeric_field_id}'")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Frequency")
        plt.show()

        # If grouped_df exists
        if 'grouped_df' in locals():
            grouped_df_sorted = grouped_df.sort_values(ascending=False)
            plt.figure(figsize=(8, 6))
            grouped_df_sorted.head(10).plot(kind='bar')
            plt.title(f"Mean of '{numeric_field_id}' by '{group_field}' (Top 10 Groups)")
            plt.xlabel(group_field)
            plt.ylabel(f"Mean of {numeric_field_id}")
            plt.tight_layout()
            plt.show()
    else:
        print("Visualization skipped: no numeric field identified.")
else:
    print("No DataFrame available for visualization.")

## 6. Conclusion
This notebook demonstrated the workflow for loading and exploring a FAIR dataset using the `mlcroissant` library, referencing all entities by their `@id`. 

- We loaded metadata and record sets.
- Extracted data from record sets into DataFrames using `@id`.
- Applied filtering, normalization, and grouping on numeric fields identified by `@id`.
- Visualized the results for basic exploratory data analysis.

For in-depth analysis, further domain-specific operations and use of field definitions and provenance within the Croissant metadata are recommended. If your dataset contains additional entities, visualizations, or complex relationships, reference their `@id` as demonstrated above.